<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L09-designing-tools-an-agent-can-use/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L09-designing-tools-an-agent-can-use/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T07-L09-designing-tools-an-agent-can-use/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T07-L09-designing-tools-an-agent-can-use/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T07-L09 · Designing tools an agent can use: the same operations as badly and as well specified tools, measured by errors and steps

**You will build:** a corrective argument validator, a linter for tool specifications, the call loop of an agent
episode, and the metrics that compare two ways of specifying the same tools. Then you measure which design
choice (names, parameters, or error text) pays for itself.

**Time:** ~75 minutes · **Runs on:** a laptop CPU, standard library only · **Prerequisites:** T07-L01-agent-loop-from-scratch, T07-L03-evaluating-agents

T07-L01 built the loop and T07-L03 measured agents with it. This lesson holds the agent fixed and changes the
tools: the same operations, specified twice. The agent here is a **scripted policy**, not a language model, so
the lesson can plant exact misreading tendencies and re-run them. What it measures is what those stated rules do
with a specification. It does not measure what any real model does.

By the end you will be able to:

1. Implement a validator that turns a bad tool call into a corrective message naming the parameter, the value received and what is accepted.
2. Implement a linter that flags vague tool names, thin descriptions and bare parameter names, and a finder for confusable tool names.
3. Implement the call loop of an agent episode and the exact metrics (success, wasted calls, silent failures, recovery) that compare toolsets.
4. Measure, with exact counts, how names, parameters and error text each change the policy's results, and which single fix pays most.
5. Explain why a result from a scripted policy is evidence about a specification's mechanics and not about a model.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T07-L01`, `T07-L03`

**This lesson assumes you can already:**

- write a Python function that loops over a list of dicts, builds a list of strings with f-strings and returns it
- read a tool definition made of a name, a description and typed parameters, as the agent loop of T07-L01 uses them
- use a regular expression with re.fullmatch, and compute exact fractions with fractions.Fraction

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Writing effective tools for agents](https://www.anthropic.com/engineering/writing-tools-for-agents) · Anthropic Engineering · *free* — the guidance on unambiguous parameter names, actionable error responses and consolidated tools that the two toolsets in this lesson are modelled on (the good descriptions are shorter than the vendor's 3-4 sentences)
- [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents) · Anthropic · *free* — where the term agent-computer interface comes from, and its appendix on prompt-engineering your tools
- [Define tools](https://platform.claude.com/docs/en/agents-and-tools/tool-use/define-tools) · Claude API documentation · *free* — what a tool definition holds (name, description, input schema) and the vendor's best practices for the description
- [Regular Expression HOWTO](https://docs.python.org/3/howto/regex.html) · Python documentation · *free* — fullmatch and character classes, for exercise 1's id pattern and exercise 2's name rule

In [ ]:
# Setup: standard library only; versions printed.
import copy
import json
import random
import re
import sys
from fractions import Fraction
from typing import Callable

print("python", sys.version.split()[0])

SEED = "T07-L09"            # every random draw below is seeded from this string plus a purpose, so a re-run is identical

_FAILED_CHECKS: list = []
_STATUS: dict = {}          # label -> "passed" | "failed" | "not started", for the progress board
_WAITING_ON: dict = {}      # label -> the stub whose NotImplementedError stopped it

# The exercises, in the order the notebook meets them: (label, name shown, stubs, where).
_EXERCISES = [("exercise 1", "check_args", ("check_args",), "section 3"),
              ("exercise 2", "lint_tool and confusable_pairs", ("lint_tool", "confusable_pairs"), "section 4"),
              ("exercise 3", "run_episode", ("run_episode",), "section 5"),
              ("exercise 4", "summarise", ("summarise",), "section 6"),
              ("exercise 5", "main_effects and best_single_fix", ("main_effects", "best_single_fix"), "section 7")]
_SHOWN = {label: shown for label, shown, _, _ in _EXERCISES}
_STUBS = {label: stubs for label, _, stubs, _ in _EXERCISES}


def _unfinished(exc: BaseException) -> str:
    """The name of the function that raised NotImplementedError: the innermost frame."""
    tb = exc.__traceback__
    while tb is not None and tb.tb_next is not None:
        tb = tb.tb_next
    return tb.tb_frame.f_code.co_name if tb is not None else ""


def _named(labels: list) -> str:
    """["exercise 3"] -> "exercise 3 (run_episode)"; several -> "exercises 3 and 4"; a demo -> its own name."""
    if len(labels) > 1 and all(label in _SHOWN for label in labels):
        nums = [label.split()[-1] for label in labels]
        return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]
    parts = [f"{label} ({_SHOWN[label]})" if label in _SHOWN else label for label in labels]
    return parts[0] if len(parts) == 1 else ", ".join(parts[:-1]) + " and " + parts[-1]


# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: it compares the process's resident memory with what it was when the
#    block started, and stops the block as soon as the growth passes `memory_mib`. After a memory
#    stop it looks every 5 ms and stops again at once only if the memory is still growing (a bare
#    `except:` swallowed the stop; clean-up that itself allocates over 32 MiB looks the same);
#    otherwise the stop repeats after 0.25 s, as a time stop does, so `finally:` blocks and
#    context managers can clean up. It looks after 0.01 s,
#    then every 0.02 s, and more often (down to every 2 ms) once the growth passes a quarter of
#    the limit or is fast enough to get there soon: numpy can add a gigabyte in a tenth of a
#    second. A loop that appends to a list reaches gigabytes in seconds, long before
#    any sensible time limit, so the time limit alone does not bound it. The limit is 768 MiB:
#    mybinder.org gives a whole notebook 2 GB, the heaviest lesson holds about 800 MiB before a
#    check starts, and a runaway check runs twice in "Run all" (its cell, then the board). The
#    largest growth a correct answer needs in any lesson is 512 MiB. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and (not state["fired"] or state.get("memory")):
            due.append(now + state.get("poll", 0.01))    # and look at the memory: first after 0.01 s, then as _fire says
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and (not state["fired"] or state.get("memory")) else 0
        if used_mib and not state["fired"]:              # close to the limit, or fast enough to come close soon
            rate = (grown - state.get("grown", 0)) / max(now - state.get("seen", started), 0.001)   # MiB a second
            state["grown"], state["seen"] = grown, now
            state["poll"] = (0.02 if grown <= memory_mib / 4 and rate * 0.02 <= (memory_mib - grown) / 4 else   # near: by
                                         max(0.002, min(0.01, (memory_mib - grown) / max(rate, 1.0) / 4)))  # a 1/4 of the time left
        if grown > memory_mib and state["next"] is not None and not state["fired"]:   # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
            state["memory"], state["at_stop"], state["poll"] = True, grown, 0.005   # then look every 5 ms
        elif state.get("memory") and state["fired"] and grown - state["at_stop"] > 32 and state["next"] is not None:
            state["next"], state["at_stop"] = now, grown        # still growing: a bare `except:` swallowed the stop
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 10     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 768      # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet says so, by name. A wrong answer prints the check's own message,
    which names the likely mistake, and the notebook carries on, so one broken exercise never hides the
    feedback on the others. A demo names the exercises it `needs`: until each has passed its check, the
    demo says which one it is waiting for and skips. Every outcome is recorded for the progress board, and
    a script run exits non-zero at the end if any check came back wrong. A check that runs longer than
    its limit is stopped and counts as failed, so a loop that never ends cannot hang the notebook.
    """
    waiting = [name for name in needs if _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped, needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        stub = _unfinished(exc)
        _STATUS[label], _WAITING_ON[label] = "not started", stub
        owner = [name for name, stubs in _STUBS.items() if stub in stubs and name != label]
        if owner:
            print(f"{label}: skipped, needs {_named(owner)} first; it still raises NotImplementedError.")
        elif label in _SHOWN:
            print(f"{label}: not implemented yet, `{stub or 'a stub'}` still raises NotImplementedError. "
                  "Fill it in, then re-run this cell.")
        else:
            print(f"{label}: skipped, `{stub or 'a stub'}` is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED, {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"



## 1. The small world: one tracker, two ways to specify the same tools

The backend is a tiny ticket tracker held in memory. It can search by words, list by person or status, read a
ticket or its comments, change a status, reassign, create a ticket or a subtask, and count by status or by person.
`OPS` lists these operations. The cell below builds the backend, the **toolsets** and the
**policy** that plays the agent. It is given: you do not need to read it to do the exercises, but you will want to
read the specifications it prints.

Each toolset has one tool per kind in `KINDS`, each tool covering two of the operations. A toolset is a choice on independent **levers**, each either `bad` or `good`:

* **names**: the tool names and descriptions (`bad`: `tickets` and `ticket`, one-line descriptions that overlap;
  `good`: `find_tickets`, `get_ticket`, and a description that says what the tool does and returns);
* **params**: the parameters (`bad`: `id`, `p`, `by`, a generic `field` and `value`, no descriptions, strings
  everywhere; `good`: `ticket_id`, `assignee`, `group_by`, each described, with enums and an id pattern);
* **errors**: what a failed call says and whether the call is checked (`bad`: the backend does not validate a person,
  status or priority, accepts a wrong value and does the wrong thing, and a rejected call says `Error: invalid request`;
  `good`: your `check_args` and the backend validate, and a message names the parameter, the value received and what
  is accepted). So this lever bundles **validation with wording**: the lesson cannot say which of the two did the work.

Crossing the levers gives every combination of bad and good, so the lesson can say which lever did the work.

**Vendor guidance, as of the dates in section 12.** Anthropic's guide to writing tools advises unambiguously named
parameters (`user_id`, not `user`) and error responses that communicate specific and actionable improvements, and
its "Define tools" page calls detailed descriptions the most important factor in tool performance and asks for at least
3 to 4 sentences per description. The `good` descriptions here are better than the `bad` ones, but they are two
sentences each, so they do not meet that last guidance, and the linter you write in exercise 2 checks only a word count.

**The policy is a script.** It reads a tool's name, description and parameters with fixed rules, and it has
**planted tendencies** that make it misread things the way a hurried model sometimes does. They are listed, with
their rates, in the cell after the world. They are assumptions written by the author of this lesson.

In [ ]:
# --- given: the backend, the five tools in two specifications, and the scripted policy ---
PEOPLE = ["ana", "ben", "chen", "dara"]
STATUSES = ["open", "in_progress", "closed"]
PRIORITIES = ["low", "medium", "high"]
GROUPS = ["status", "assignee"]
ENUMS = {"person": PEOPLE, "status": STATUSES, "priority": PRIORITIES, "group": GROUPS, "@field": ["status", "assignee"]}
TITLES = ["login timeout", "export csv", "dark mode", "search lag", "email bounce", "invoice rounding",
          "upload crash", "audit log", "cache stale", "password reset", "report layout", "api limits"]
NEW_TITLES = ["fix typo in footer", "add retry to uploader", "document the api", "remove old banner",
              "speed up dashboard", "rename export button", "clean up logs", "check mobile layout"]
COMMENTS = ["reproduced on staging", "waiting on the customer", "needs a design review", "fixed in the next release"]
OPS = ["search_text", "list_filtered", "read_ticket", "read_comments", "set_status", "reassign",
       "create", "create_subtask", "count_by_status", "count_by_assignee"]
KINDS = ["find", "get", "update", "create", "stats"]
OP_KIND = dict(zip(OPS, ["find", "find", "get", "get", "update", "update", "create", "create", "stats", "stats"]))
# What a user says -> the value a tool accepts. The policy reads these words; tools do not.
WORDS = {"status": {"done": "closed", "finished": "closed", "started": "in_progress", "underway": "in_progress",
                    "new": "open", "reopened": "open"},
         "priority": {"urgent": "high", "critical": "high", "minor": "low", "trivial": "low",
                      "normal": "medium", "standard": "medium"},
         "group": {"state": "status", "progress": "status", "owner": "assignee", "person": "assignee"}}
NEED = {"search_text": "find tickets by words in the title",
        "list_filtered": "find tickets of one person or with one status",
        "read_ticket": "read one ticket", "read_comments": "read the comments on a ticket",
        "set_status": "change the status of a ticket", "reassign": "change the person a ticket is assigned to",
        "create": "create a new ticket", "create_subtask": "create a subtask of a ticket",
        "count_by_status": "count tickets by status", "count_by_assignee": "count tickets by person"}
TASKS_PER_OP = 4
MAX_STEPS = 6
SLIP = Fraction(1, 4)

TOOL_TEXT = {   # names lever: (name, description) for each kind
    "bad": {"find": ("tickets", "Gets tickets."), "get": ("ticket", "Gets a ticket."), "update": ("edit", "Edits a ticket."),
            "create": ("new", "Makes a ticket."), "stats": ("summary", "Gets a summary of tickets.")},
    "good": {"find": ("find_tickets", "Find tickets by words in the title, or list the tickets of one person or with one "
                                      "status. Returns the matching ids with their titles, and says so when nothing matches."),
             "get": ("get_ticket", "Read one ticket by its id: title, status, person and priority. Set include_comments "
                                   "to read the comments on the ticket instead."),
             "update": ("update_ticket", "Change the status of one ticket, or change the person it is assigned to. "
                                         "The ticket must already exist."),
             "create": ("create_ticket", "Create a new ticket with a title and a priority, or a subtask of an existing "
                                         "ticket by giving the id of its parent."),
             "stats": ("count_tickets", "Count tickets grouped by status or by person. Returns one count per group.")}}


def _p(name, slot, type_, description, required, **extra):
    return dict(name=name, slot=slot, type=type_, description=description, required=required, **extra)


_ID = dict(pattern=r"T\d{3}", example="T004")
PARAM_LISTS = {   # params lever: the parameters of each kind
    "bad": {"find": [_p("q", "text", "string", "", False), _p("who", "person", "string", "", False), _p("state", "status", "string", "", False)],
            "get": [_p("id", "ticket", "string", "", True), _p("extra", "comments", "boolean", "", False)],
            "update": [_p("id", "ticket", "string", "", True), _p("field", "@field", "string", "", True), _p("value", "@value", "string", "", True)],
            "create": [_p("name", "title", "string", "", True), _p("p", "priority", "string", "", True), _p("parent", "parent", "string", "", False)],
            "stats": [_p("by", "group", "string", "", True)]},
    "good": {"find": [_p("text", "text", "string", "Words to look for in ticket titles.", False),
                      _p("assignee", "person", "string", "Only tickets assigned to this person.", False, enum=PEOPLE),
                      _p("status", "status", "string", "Only tickets with this status.", False, enum=STATUSES)],
             "get": [_p("ticket_id", "ticket", "string", "The id of the ticket, like T004.", True, **_ID),
                     _p("include_comments", "comments", "boolean", "True to read the comments on the ticket instead of its fields.", False)],
             "update": [_p("ticket_id", "ticket", "string", "The id of the ticket, like T004.", True, **_ID),
                        _p("status", "status", "string", "The new status of the ticket.", False, enum=STATUSES),
                        _p("assignee", "person", "string", "The person to give the ticket to.", False, enum=PEOPLE)],
             "create": [_p("title", "title", "string", "A short title for the new ticket.", True),
                        _p("priority", "priority", "string", "How urgent the ticket is.", True, enum=PRIORITIES),
                        _p("parent_id", "parent", "string", "The id of the parent ticket, like T004, to make a subtask.", False, **_ID)],
             "stats": [_p("group_by", "group", "string", "What to group the counts by.", True, enum=GROUPS)]}}
ALIASES = {"text": {"text", "query", "search", "word"}, "person": {"person", "assignee", "who", "owner", "user"},
           "status": {"status", "state"}, "ticket": {"ticket", "id"}, "title": {"title", "name"},
           "priority": {"priority", "urgency"}, "parent": {"parent"}, "group": {"group", "by", "grouped"},
           "comments": {"comment", "comments", "include"}}
SYNONYMS = {"edit": "change", "update": "change", "set": "change", "new": "create", "make": "create", "add": "create",
            "summary": "count", "stats": "count", "total": "count", "search": "find", "list": "find", "lookup": "find",
            "get": "read", "show": "read"}
STOPWORDS = {"a", "an", "the", "of", "by", "in", "to", "or", "and", "one", "it", "its", "is", "for", "with", "on", "at", "as",
             "from", "that", "this", "are", "be", "can", "returns", "return", "use", "uses", "instead", "when"}


def initial_world() -> dict:
    rng = random.Random(f"{SEED}:state")
    tickets = {}
    for i, title in enumerate(TITLES, 1):
        tickets[f"T{i:03d}"] = {"title": title, "status": rng.choice(STATUSES), "person": rng.choice(PEOPLE),
                                "priority": rng.choice(PRIORITIES), "parent": "",
                                "comments": [rng.choice(COMMENTS) for _ in range(rng.randint(0, 2))]}
    return {"tickets": tickets, "executed": []}


INITIAL = initial_world()


def fresh_world() -> dict:
    """A private copy of the starting tracker: every episode gets its own."""
    return copy.deepcopy(INITIAL)


class BackendError(Exception):
    def __init__(self, kind, slot, value=None):
        super().__init__(kind)
        self.kind, self.slot, self.value = kind, slot, value


def _ticket(world, value, slot):
    if not isinstance(value, str) or not re.fullmatch(r"T\d{3}", value):
        raise BackendError("ticket_format", slot, value)
    if value not in world["tickets"]:
        raise BackendError("ticket_missing", slot, value)
    return world["tickets"][value]


def _enum(slot, value):
    if value not in ENUMS[slot]:
        raise BackendError("enum", slot, value)


def _log(world, op, **args):
    world["executed"].append((op, tuple(sorted(args.items()))))


def _new_id(world):
    return f"T{len(world['tickets']) + 1:03d}"


def run_backend(kind: str, world: dict, a: dict, strict: bool):
    """Do what the arguments ask, on the tracker. `a` maps slot names to values. `strict` is the good-error behaviour."""
    T = world["tickets"]
    if kind == "find":
        text, person, status = a.get("text"), a.get("person"), a.get("status")
        if text is not None:
            found = sorted(i for i, t in T.items() if str(text).lower() in t["title"]); _op = ("search_text", {"text": text})
        elif person is not None or status is not None:
            if strict:
                if person is not None: _enum("person", person)
                if status is not None: _enum("status", status)
            found = sorted(i for i, t in T.items() if (person is None or t["person"] == person) and (status is None or t["status"] == status))
            _op = ("list_filtered", {k: v for k, v in (("person", person), ("status", status)) if v is not None})
        else:
            raise BackendError("noargs", "")
        if not found and strict:
            raise BackendError("empty", "", None)
        _log(world, _op[0], **_op[1])
        return [(i, T[i]["title"]) for i in found]
    if kind == "get":
        if a.get("ticket") is None:
            raise BackendError("noargs", "")
        t = _ticket(world, a["ticket"], "ticket")
        if a.get("comments") is True:
            _log(world, "read_comments", ticket=a["ticket"]); return list(t["comments"])
        _log(world, "read_ticket", ticket=a["ticket"]); return dict(t)
    if kind == "update":
        a = dict(a)
        if "@field" in a:
            field, value = a.pop("@field"), a.pop("@value", None)
            _enum("@field", field)
            a["status" if field == "status" else "person"] = value
        t = _ticket(world, a.get("ticket"), "ticket") if a.get("ticket") is not None else None
        if t is None or ("status" in a) == ("person" in a):
            raise BackendError("noargs", "")
        slot = "status" if "status" in a else "person"
        _enum(slot, a[slot])
        t[slot] = a[slot]
        _log(world, "set_status" if slot == "status" else "reassign", ticket=a["ticket"], **{slot: a[slot]})
        return {"ticket": a["ticket"], slot: a[slot]}
    if kind == "create":
        title, priority, parent = a.get("title"), a.get("priority"), a.get("parent")
        if not title or priority is None:
            raise BackendError("noargs", "")
        if strict or priority in PRIORITIES:
            _enum("priority", priority)
        if parent is not None:
            _ticket(world, parent, "parent")
        tid = _new_id(world)
        T[tid] = {"title": title, "status": "open", "person": "", "priority": priority, "parent": parent or "", "comments": []}
        if parent is not None:
            _log(world, "create_subtask", parent=parent, priority=priority, title=title)
        else:
            _log(world, "create", priority=priority, title=title)
        return {"created": tid}
    group = a.get("group")
    _enum("group", group)
    key = "status" if group == "status" else "person"
    counts = {}
    for t in T.values():
        counts[t[key]] = counts.get(t[key], 0) + 1
    _log(world, "count_by_" + group)
    return dict(sorted(counts.items()))


class Toolset:
    """Five tools. `names`, `params` and `errors` are each "bad" or "good". `.specs` maps tool name -> specification
    ({"name", "description", "params": [...]}); `.call(world, name, args)` returns (ok, result_or_message)."""

    def __init__(self, names: str, params: str, errors: str):
        self.levers = (names, params, errors)
        self.errors = errors
        self.specs, self.kind_of, self.slot_of, self.order = {}, {}, {}, []
        self.name_of = {}
        for kind in KINDS:
            name, description = TOOL_TEXT[names][kind]
            plist = PARAM_LISTS[params][kind]
            self.specs[name] = {"name": name, "description": description,
                                "params": [{k: v for k, v in p.items() if k != "slot"} for p in plist]}
            self.kind_of[name], self.name_of[kind] = kind, name
            self.slot_of[name] = {p["name"]: p["slot"] for p in plist}
            self.order.append(name)

    def _vague_problem(self, spec, args) -> bool:
        names = [p["name"] for p in spec["params"]]
        if any(k not in names for k in args):
            return True
        for p in spec["params"]:
            v = args.get(p["name"])
            if v is None:
                if p["required"]:
                    return True
                continue
            if (p["type"] == "string") != isinstance(v, str) or (p["type"] == "boolean") != isinstance(v, bool):
                return True
        return False

    def call(self, world: dict, name: str, args: dict):
        if name not in self.specs:
            return False, "Error: unknown tool" if self.errors == "bad" else f"Error: no tool named {name!r}; the tools are [{', '.join(self.order)}]"
        spec, kind = self.specs[name], self.kind_of[name]
        if self.errors == "good":
            problems = check_args(spec, args)
            if problems:
                return False, "Error: " + " | ".join(problems)
        elif self._vague_problem(spec, args):
            return False, "Error: invalid request"
        slots = {self.slot_of[name][p]: v for p, v in args.items() if p in self.slot_of[name]}
        try:
            return True, run_backend(kind, world, slots, self.errors == "good")
        except BackendError as err:
            return False, self._explain(name, err)

    def _param_of(self, name, slot):
        by_slot = {s: p for p, s in self.slot_of[name].items()}
        return by_slot.get(slot) or by_slot.get("@value") or slot

    def _explain(self, name, err) -> str:
        if self.errors == "bad":
            return "Error: invalid request"
        p = self._param_of(name, err.slot)
        if err.kind == "enum":
            return f"Error: parameter '{p}' must be one of [{', '.join(ENUMS[err.slot])}]; got {err.value!r}"
        if err.kind == "ticket_format":
            return (f"Error: parameter '{p}' must look like T004; got {err.value!r}. "
                    f"To search tickets by words call {self.name_of['find']}")
        if err.kind == "ticket_missing":
            return f"Error: no ticket {err.value!r}; ids run from T001 to T{len(INITIAL['tickets']):03d}"
        if err.kind == "empty":
            return f"Error: no ticket matched. To read one ticket by its id call {self.name_of['get']}"
        return f"Error: nothing to do; give at least one of [{', '.join(q['name'] for q in self.specs[name]['params'])}]"


def build_tasks() -> list:
    """The tasks, a few per operation, each with the user's wording, the canonical values and the expected effect."""
    tasks = []
    inverse = {slot: {} for slot in WORDS}
    for slot, table in WORDS.items():
        for word, value in table.items():
            inverse[slot].setdefault(value, []).append(word)
    ids = sorted(INITIAL["tickets"])
    T = INITIAL["tickets"]
    for oi, op in enumerate(OPS):
        for j in range(TASKS_PER_OP):
            rng = random.Random(f"{SEED}:task:{op}:{j}")
            wording, canon = {}, {}

            def ticket(slot, pool):
                tid = rng.choice(pool); n = int(tid[1:])
                wording[slot], canon[slot] = rng.choice([str(n), f"#{n}", f"t{n}"]), tid
                return tid

            def word(slot, value):
                wording[slot], canon[slot] = rng.choice(inverse[slot][value]), value

            if op == "search_text":
                w = rng.choice(TITLES).split()[0]; wording["text"] = canon["text"] = w
            elif op == "list_filtered":
                if j % 2 == 0:
                    p = rng.choice(sorted({t["person"] for t in T.values()})); wording["person"], canon["person"] = p.capitalize(), p
                else:
                    s = rng.choice(sorted({t["status"] for t in T.values()})); word("status", s)
            elif op in ("read_ticket", "read_comments"):
                tid = ticket("ticket", [i for i in ids if T[i]["comments"]] if op == "read_comments" else ids)
                if op == "read_comments":
                    wording["comments"] = canon["comments"] = True
            elif op == "set_status":
                tid = ticket("ticket", ids); word("status", rng.choice([s for s in STATUSES if s != T[tid]["status"]]))
            elif op == "reassign":
                tid = ticket("ticket", ids); p = rng.choice([x for x in PEOPLE if x != T[tid]["person"]])
                wording["person"], canon["person"] = p.capitalize(), p
            elif op in ("create", "create_subtask"):
                wording["title"] = canon["title"] = NEW_TITLES[(oi * TASKS_PER_OP + j) % len(NEW_TITLES)]
                word("priority", rng.choice(PRIORITIES))
                if op == "create_subtask":
                    ticket("parent", ids)
            else:
                word("group", "status" if op == "count_by_status" else "assignee")
            args = tuple(sorted((k, v) for k, v in canon.items() if k not in ("group", "comments")))
            world = fresh_world()
            a = dict(canon)
            kind = OP_KIND[op]
            if kind == "stats":
                a = {"group": canon["group"]}
            if kind == "get" and "comments" not in a:
                a.pop("comments", None)
            run_backend(kind, world, a, True)
            assert world["executed"][-1][0] == op, (op, world["executed"])
            tasks.append({"id": f"task-{len(tasks) + 1:02d}", "op": op, "wording": wording, "canon": canon,
                          "expected": world["executed"][-1], "after": world["tickets"]})
    return tasks


TASKS = build_tasks()


def judge(world: dict, task: dict) -> bool:
    """True when the last thing the tracker did is exactly the expected operation with the expected arguments and
    the tracker ends in the expected state. A tool that accepted a wrong value fails here without ever raising an error."""
    done = world["executed"]
    return bool(done) and done[-1] == task["expected"] and world["tickets"] == task["after"]


def _tokens(text: str) -> set:
    out = set()
    for w in re.findall(r"[a-z]+", text.lower()):
        if w in STOPWORDS:
            continue
        if len(w) > 3 and w.endswith("s") and w != "status":
            w = w[:-1]
        out.add(SYNONYMS.get(w, w))
    return out


def tool_score(spec: dict, op: str) -> int:
    need = _tokens(NEED[op])
    return 2 * len(need & _tokens(spec["name"].replace("_", " "))) + len(need & _tokens(spec["description"]))


def choose_option(slot: str, wording: str, options: list):
    """The accepted value a user's wording stands for, or None."""
    w = str(wording).lower()
    target = WORDS.get(slot, {}).get(w, w)
    if target in options:
        return target
    return next((o for o in options if o in ALIASES.get(slot, ())), None)


def _format_ticket(wording: str) -> str:
    digits = re.findall(r"\d+", str(wording))
    return f"T{int(digits[-1]):03d}" if digits else wording


class ScriptedPolicy:
    """The agent: a script with planted tendencies (see TENDENCIES). `propose(calls)` returns (tool name, args) or None."""

    def __init__(self, toolset: Toolset, task: dict, rng: random.Random):
        self.ts, self.task, self.rng = toolset, task, rng
        self.tool, self.tried, self.ranking, self.args, self.assign = None, [], [], {}, {}

    def propose(self, calls: list):
        if not calls:
            scores = {n: tool_score(self.ts.specs[n], self.task["op"]) for n in self.ts.order}
            best = max(scores.values())
            first = self.rng.choice([n for n in self.ts.order if scores[n] == best])
            rest = sorted((n for n in self.ts.order if n != first), key=lambda n: (-scores[n], self.ts.order.index(n)))
            self.ranking, self.tool = [first] + rest, first
            self._fill()
        elif calls[-1]["ok"]:
            return None
        else:
            self._react(calls[-1]["text"])
        return self.tool, dict(self.args)

    def _wishes(self, p: dict) -> list:
        wishes = self.task["wording"]
        by_name = [s for s in sorted(wishes) if _tokens_plain(p["name"]) & ALIASES[s]]
        return by_name or [s for s in sorted(wishes) if _tokens_plain(p["description"]) & ALIASES[s]]

    def _value(self, slot: str, wording, p: dict):
        if slot == "comments":
            return True
        if p.get("enum"):
            canon = choose_option(slot, wording, p["enum"]) or wording
        elif p.get("pattern"):
            canon = _format_ticket(wording)
        else:
            return wording
        return wording if self.rng.random() < SLIP else canon

    def _fill(self):
        params, wishes = self.ts.specs[self.tool]["params"], self.task["wording"]
        names = [p["name"] for p in params]
        kv = "field" in names and "value" in names
        fits = {p["name"]: self._wishes(p) for p in params if not (kv and p["name"] in ("field", "value"))}
        assign, claimed = {}, set()
        for n, m in fits.items():
            if len(m) == 1 and m[0] not in claimed:
                assign[n] = m[0]; claimed.add(m[0])
        for n, m in fits.items():
            free = [s for s in m if s not in claimed]
            if n not in assign and len(m) > 1 and free:
                assign[n] = self.rng.choice(free); claimed.add(assign[n])
        if kv:
            free = [s for s in sorted(wishes) if s not in claimed]
            if free:
                assign["field"], assign["value"] = "@" + free[0], "@" + free[0]; claimed.add(free[0])
        for p in sorted(params, key=lambda q: not q["required"]):
            n = p["name"]
            free = [s for s in sorted(wishes) if s not in claimed]
            if n in fits and n not in assign and free:
                assign[n] = self.rng.choice(free); claimed.add(assign[n])
        self.args, self.assign = {}, {}
        for p in params:
            n = p["name"]
            if n not in assign:
                continue
            if assign[n].startswith("@"):
                slot = assign[n][1:]
                if n == "field":
                    self.assign[n] = (slot, slot)
                    self.args[n] = choose_option(slot, slot, p["enum"]) if p.get("enum") else slot
                else:
                    self.assign[n] = (slot, wishes[slot]); self.args[n] = wishes[slot]
                continue
            self.assign[n] = (assign[n], wishes[assign[n]])
            self.args[n] = self._value(assign[n], wishes[assign[n]], p)

    def _react(self, text: str):
        fixed = False
        for m in re.finditer(r"parameter '(\w+)' must be one of \[([^\]]*)\]; got", text):
            if m.group(1) in self.assign:
                slot, w = self.assign[m.group(1)]
                c = choose_option(slot, w, [o.strip() for o in m.group(2).split(",")])
                if c is not None:
                    self.args[m.group(1)] = c; fixed = True
        for m in re.finditer(r"parameter '(\w+)' must look like", text):
            if m.group(1) in self.assign and re.search(r"\d", str(self.assign[m.group(1)][1])):
                self.args[m.group(1)] = _format_ticket(self.assign[m.group(1)][1]); fixed = True
        for m in re.finditer(r"unknown parameter '(\w+)'", text):
            fixed = self.args.pop(m.group(1), None) is not None or fixed
        for m in re.finditer(r"missing required parameter '(\w+)'", text):
            used = {s for s, _ in self.assign.values()}
            free = [s for s in sorted(self.task["wording"]) if s not in used]
            if free and m.group(1) not in self.args:
                self.assign[m.group(1)] = (free[0], self.task["wording"][free[0]]); self.args[m.group(1)] = self.task["wording"][free[0]]; fixed = True
        if fixed:
            return
        m = re.search(r"call (\w+)", text)
        if m and m.group(1) in self.ts.specs and m.group(1) != self.tool:
            self.tried.append(self.tool); self.tool = m.group(1); self._fill(); return
        act = self.rng.choice(["retry", "re-guess", "next tool"])
        if act == "re-guess":
            self._fill()
        elif act == "next tool":
            self.tried.append(self.tool)
            left = [n for n in self.ranking if n not in self.tried]
            if left:
                self.tool = left[0]; self._fill()


def _tokens_plain(text: str) -> set:
    return set(re.findall(r"[a-z]+", text.lower()))


def make_policy(toolset: Toolset, task: dict) -> ScriptedPolicy:
    """One policy per episode, seeded from the toolset's levers and the task id, so no episode depends on another."""
    return ScriptedPolicy(toolset, task, random.Random(f"{SEED}:{toolset.levers}:{task['id']}"))


TENDENCIES = [
    ("tie", "two tools read equally well for the job", "a coin picks one: no preference for the right one"),
    ("guess", "a parameter whose name fits no wish", "it takes any wish not yet placed, at random"),
    ("raw", "no enum or id pattern in the specification", "it sends the user's own wording (\"Ana\", \"done\", \"#12\")"),
    ("slip", f"a value it could format correctly", f"it sends the raw wording anyway, {SLIP.numerator} time in {SLIP.denominator}"),
    ("key/value", "a generic `field` and `value` pair", "it puts its own name for the wish in `field`"),
    ("flail", "an error that says nothing useful", "it retries, re-guesses or tries the next tool, by coin"),
    ("trust", "a call that returns without an error", "it stops: it cannot tell a wrong answer from a right one"),
]

## 2. Read the two specifications and the policy's tendencies

Run the next cell. It prints the all-bad and the all-good toolset side by side, the policy's tendencies with their
rates, and a few of the tasks with the wording a user would use. Nothing in it is yours to write.

In [ ]:
# --- given: read the two specifications, then the policy's stated tendencies ---
def show_specs() -> None:
    for levers in (("bad", "bad", "bad"), ("good", "good", "good")):
        ts = Toolset(*levers)
        print(f"== names={levers[0]}, params={levers[1]} ==")
        for name in ts.order:
            spec = ts.specs[name]
            plist = ", ".join(f"{p['name']}:{p['type']}{'*' if p['required'] else ''}" + (f" in {p['enum']}" if p.get("enum") else "")
                              for p in spec["params"])
            print(f"  {name}({plist})\n      {spec['description']}")
    print("\n(* marks a required parameter)")
    print(f"\nthe policy's planted tendencies ({len(TENDENCIES)}):")
    for tag, when, what in TENDENCIES:
        print(f"  {tag:<10} when: {when:<46} then: {what}")
    print(f"\n{len(TASKS)} tasks, {TASKS_PER_OP} per operation; each episode may make at most {MAX_STEPS} calls")
    for t in TASKS[::8]:
        print(f"  {t['id']} {t['op']:<17} the user says {t['wording']}")


if __name__ == "__main__":
    _try("specification tour", show_specs)

## 3. Exercise 1: an error that tells the agent what to do

Anthropic's engineering guide on writing tools (11 September 2025; section 12 gives the date it was read) asks for error
responses that communicate specific and actionable improvements instead of opaque codes. The policy reads
one fixed shape of message, so your messages must have exactly that shape.

**`check_args(spec, args)`** returns the list of problems with a call, each a string, in this order:

1. for each name in `args` that is not a parameter of the spec, in sorted order:
   `unknown parameter 'NAME'; this tool takes [a, b]` (the parameter names in declared order, joined by `, `);
2. then for each parameter of the spec in declared order, at most one problem each:
   * absent, or `None`, and `required`: `missing required parameter 'NAME' (TYPE): DESCRIPTION`;
   * absent and not required: no problem (a `None` value counts as absent here too, so `None` on an optional parameter is no problem and no type error);
   * wrong type (`string` needs a `str`, `integer` an `int` that is not a `bool`, `boolean` a `bool`):
     `parameter 'NAME' must be a string; got VALUE` (`an integer` for integer; `a boolean` for boolean), where VALUE is `repr(value)`;
   * a type-correct value not in `enum` (when the parameter has one):
     `parameter 'NAME' must be one of [a, b, c]; got VALUE`;
   * a type-correct string that does not match `pattern` in full (`re.fullmatch`), when it has one:
     `parameter 'NAME' must look like EXAMPLE; got VALUE`.

A call with nothing wrong returns `[]`. Never change `spec` or `args`.

<details><summary>💡 Hint 1 — what to think about</summary>

Unknown names come first and across the whole call; the per-parameter problems follow the order the spec declares,
not the order of the call. A parameter gets one problem at most: which of the checks makes the next one pointless?
And is `True` an `int` in Python?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Build a list. Loop over `sorted(args)` for the unknown ones, then over `spec["params"]`. Test the type with a small
table from the type word to a test and to its article. Use `repr(value)` inside an f-string for the value.

</details>

In [ ]:
def check_args(spec: dict, args: dict) -> list:
    """The problems with a call to one tool, as the corrective messages described above.

    Example:
        >>> spec = {"name": "t", "description": "d", "params": [
        ...     {"name": "n", "type": "integer", "description": "How many.", "required": True}]}
        >>> check_args(spec, {"n": "3"})
        ["parameter 'n' must be an integer; got '3'"]

    Returns:
        A list of strings; empty when the call is fine.
    """
    # YOUR CODE HERE
    raise NotImplementedError


_PROBE = {"name": "probe", "description": "d", "params": [
    {"name": "ticket_id", "type": "string", "description": "The id, like T004.", "required": True, "pattern": r"T\d{3}", "example": "T004"},
    {"name": "status", "type": "string", "description": "The status.", "required": False, "enum": ["open", "closed"]},
    {"name": "limit", "type": "integer", "description": "How many.", "required": False},
    {"name": "full", "type": "boolean", "description": "All fields.", "required": False}]}


def _check_args() -> None:
    assert check_args(_PROBE, {"ticket_id": "T004", "status": "open", "limit": 3, "full": False}) == [], (
        "a correct call has no problems: return an empty list, and treat a False boolean as present")
    got = check_args(_PROBE, {})
    assert got == ["missing required parameter 'ticket_id' (string): The id, like T004."], f"got {got}: name, type in brackets, then the description"
    assert check_args(_PROBE, {"ticket_id": "T004", "status": "done"}) == [
        "parameter 'status' must be one of [open, closed]; got 'done'"], "name the parameter, list the accepted values, show the value received with repr"
    assert check_args(_PROBE, {"ticket_id": "#4"}) == ["parameter 'ticket_id' must look like T004; got '#4'"], (
        "a pattern that does not match in full: show the example, not the regular expression")
    assert check_args(_PROBE, {"ticket_id": "T004", "limit": "3"}) == ["parameter 'limit' must be an integer; got '3'"], "the article is part of the message"
    assert check_args(_PROBE, {"ticket_id": "T004", "limit": True}) == ["parameter 'limit' must be an integer; got True"], (
        "a bool is not an integer here, although Python says it is a subclass of int")
    got = check_args(_PROBE, {"zeta": 1, "alpha": 2, "ticket_id": "T004"})
    assert got == ["unknown parameter 'alpha'; this tool takes [ticket_id, status, limit, full]",
                   "unknown parameter 'zeta'; this tool takes [ticket_id, status, limit, full]"], f"got {got}: unknown names first, sorted, each listing the real parameters"
    print("exercise 1 looks right")


if __name__ == "__main__":
    _try("exercise 1", _check_args)

## 4. Exercise 2: lint a specification before an agent ever sees it

Some defects show in the text of a specification. A linter cannot tell you a tool is good, but it can tell you
cheaply that one is probably bad, and it needs no agent to run.

**`lint_tool(spec)`** returns the sorted list of codes that fire, each once:

* `unclear-name`: the name is not `verb_noun` form: it does not match `^(get|find|list|read|create|update|delete|count)_[a-z]+(_[a-z]+)*$`;
* `thin-description`: the description has fewer than `MIN_WORDS` words (`str.split()`);
* `bare-param`: some parameter's name is in `BARE_NAMES`;
* `undescribed-param`: some parameter's description is empty or only whitespace;
* `unchecked-id`: some parameter named `id` or ending in `_id` has no `pattern`.

**`confusable_pairs(specs)`** takes a list of specs and returns the list of `(name_a, name_b)` pairs, `a` before `b`
in the list, whose names are confusable: split each name on `_`, drop one trailing `s` from each piece longer than
three letters, and call two names confusable when one set of pieces is a subset of the other (equal sets included).

<details><summary>💡 Hint 1 — what to think about</summary>

Each code is an independent test of one part of the spec, so build the result as a set. For the pairs, think about
which pairs of positions to visit so that no pair appears twice and `a` always comes first.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

A regular expression for the name; `len(description.split())` for the words; `any(...)` over the parameters for
the rest. For the pairs, turn each name into a frozenset of pieces and compare with `<=` for every `i < j`.

</details>

In [ ]:
MIN_WORDS = 8
BARE_NAMES = ("id", "q", "p", "by", "value", "field", "extra", "data", "input", "key")


def lint_tool(spec: dict) -> list:
    """The sorted codes that fire on one tool specification (see the rules above).

    Example:
        >>> lint_tool({"name": "get_ticket", "description": "Read one ticket by its id and return every field it has.",
        ...            "params": [{"name": "ticket_id", "type": "string", "description": "The id.", "required": True,
        ...                        "pattern": "T[0-9]+", "example": "T4"}]})
        []

    Returns:
        A sorted list of codes, each at most once.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def confusable_pairs(specs: list) -> list:
    """Pairs of tool names that could be mistaken for each other (see the rule above).

    Example:
        >>> confusable_pairs([{"name": "ticket"}, {"name": "tickets"}, {"name": "get_ticket"}])
        [('ticket', 'tickets'), ('ticket', 'get_ticket'), ('tickets', 'get_ticket')]

    Returns:
        A list of (name_a, name_b) tuples in list order.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _s(name="get_ticket", description="Read one ticket by its id and return every field it has.", params=()):
    return {"name": name, "description": description, "params": list(params)}


def _check_lint() -> None:
    ok_param = {"name": "ticket_id", "type": "string", "description": "The id.", "required": True, "pattern": "T[0-9]+", "example": "T4"}
    assert lint_tool(_s(params=[ok_param])) == [], "a well-formed specification fires nothing"
    assert lint_tool(_s(name="ticket")) == ["unclear-name"], "a bare noun is not verb_noun"
    assert lint_tool(_s(name="fetch_ticket")) == ["unclear-name"], "only the listed verbs count"
    assert lint_tool(_s(description="Gets a ticket.")) == ["thin-description"], "fewer than MIN_WORDS words"
    assert lint_tool(_s(description="one two three four five six seven eight")) == [], "exactly MIN_WORDS words is enough"
    bare = dict(ok_param, name="q", description="Query.")
    assert lint_tool(_s(params=[bare])) == ["bare-param"], "a bare name, described, with no id in it"
    assert lint_tool(_s(params=[dict(ok_param, description="  ")])) == ["undescribed-param"], "whitespace is not a description"
    assert lint_tool(_s(params=[{k: v for k, v in ok_param.items() if k != "pattern"}])) == ["unchecked-id"], "an _id parameter with no pattern"
    both = dict(ok_param, name="id", description="", pattern=None)
    got = lint_tool(_s(name="x", description="", params=[both]))
    assert got == ["bare-param", "thin-description", "unchecked-id", "unclear-name", "undescribed-param"], f"got {got}: sorted, each code once"
    pairs = confusable_pairs([_s("ticket"), _s("tickets"), _s("get_ticket"), _s("summary")])
    assert pairs == [("ticket", "tickets"), ("ticket", "get_ticket"), ("tickets", "get_ticket")], f"got {pairs}: strip one trailing s, then test subset in both directions"
    assert confusable_pairs([_s("find_tickets"), _s("get_ticket"), _s("count_tickets")]) == [], "sharing a piece is not enough: one set must contain the other"
    assert confusable_pairs([_s("status"), _s("stat")]) == [], "pieces of three letters or fewer keep their s; `status` becomes `statu`"
    print("exercise 2 looks right")


if __name__ == "__main__":
    _try("exercise 2", _check_lint)

In [ ]:
def show_lint() -> None:
    for levers in (("bad", "bad", "bad"), ("good", "good", "good")):
        ts = Toolset(*levers)
        print(f"names={levers[0]} params={levers[1]}")
        for name in ts.order:
            print(f"  {name:<14}{lint_tool(ts.specs[name])}")
        print("  confusable:", confusable_pairs([ts.specs[n] for n in ts.order]) or "none")
    total = {lv: sum(len(lint_tool(s)) for s in Toolset(lv, lv, "bad").specs.values()) for lv in ("bad", "good")}
    print(f"findings in all: bad names and params {total['bad']}, good names and params {total['good']}")


if __name__ == "__main__":
    _try("lint demo", show_lint, needs=("exercise 2",))

## 5. Exercise 3: the loop that runs one episode

You wrote this loop in T07-L01. Here it has one job: let the policy and the toolset exchange calls until the
episode ends, and keep an honest record of what happened.

**`run_episode(policy, toolset, task, max_steps)`** starts from `fresh_world()` and repeats, at most `max_steps`
times: ask `policy.propose(calls)` (passing the list of records so far) for the next call; `None` ends the
episode; otherwise call `toolset.call(world, tool_name, args)`, which returns `(ok, text)`, and append a record
`{"tool": ..., "args": ..., "ok": ..., "text": ...}`; an `ok` call also ends the episode (the policy stops at the
first call that returns without an error). It returns a dict with:

* `calls`: the records, in order; `steps`: how many calls were made; `errors`: how many of them were not `ok`;
* `stopped_ok`: whether the last call was `ok` (`False` when there were no calls);
* `success`: `judge(world, task)`, taken after the loop.

<details><summary>💡 Hint 1 — what to think about</summary>

Three things end the loop, and only one of them is a call that succeeded. What is `stopped_ok` when the policy
gives up before making any call? And when the episode ends because the step budget ran out on an error?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

A `for` over `range(max_steps)`; inside, `propose`, a `break` on `None`, the call, the append, a `break` on `ok`.
Count the errors from the records after the loop rather than keeping a second counter that can drift.

</details>

In [ ]:
def run_episode(policy, toolset, task: dict, max_steps: int) -> dict:
    """One episode of `policy` using `toolset` on `task`, as described above.

    Example (a policy that makes one call and stops on its success):
        >>> ep = run_episode(policy, toolset, task, 6)    # doctest: +SKIP
        >>> sorted(ep)
        ['calls', 'errors', 'stopped_ok', 'steps', 'success']

    Returns:
        A dict with keys calls, steps, errors, stopped_ok and success.
    """
    # YOUR CODE HERE
    raise NotImplementedError


class _Scripted:
    """A policy that replays a list of calls, then gives up (used only by the check)."""

    def __init__(self, script):
        self.script, self.seen = list(script), []

    def propose(self, calls):
        self.seen.append(len(calls))
        return self.script[len(calls)] if len(calls) < len(self.script) else None


def _check_episode() -> None:
    ts = Toolset("good", "good", "good")
    task = next(t for t in TASKS if t["op"] == "read_ticket")
    tid = task["canon"]["ticket"]
    right = ("get_ticket", {"ticket_id": tid})
    wrong = ("get_ticket", {"ticket_id": "4"})
    pol = _Scripted([wrong, right, right])
    ep = run_episode(pol, ts, task, 6)
    assert ep["steps"] == 2 and ep["errors"] == 1 and ep["stopped_ok"] and ep["success"], (
        f"got {ep['steps']} steps, {ep['errors']} errors: the first call fails, the second succeeds and ends the episode")
    assert pol.seen == [0, 1], f"propose saw {pol.seen}: it is given the records so far, so its argument grows by one each step"
    assert [c["ok"] for c in ep["calls"]] == [False, True] and ep["calls"][0]["tool"] == "get_ticket" and ep["calls"][0]["args"] == wrong[1], (
        "each record holds the tool, the args as sent, ok and the text")
    ep = run_episode(_Scripted([wrong] * 9), ts, task, 3)
    assert ep["steps"] == 3 and ep["errors"] == 3 and not ep["stopped_ok"] and not ep["success"], "the step budget ends the episode"
    ep = run_episode(_Scripted([]), ts, task, 3)
    assert ep["steps"] == 0 and ep["calls"] == [] and ep["stopped_ok"] is False and ep["success"] is False, "a policy that makes no call has not stopped on a success"
    other = next(t for t in TASKS if t["op"] == "set_status")
    a = ("update_ticket", {"ticket_id": other["canon"]["ticket"], "status": other["canon"]["status"]})
    first = run_episode(_Scripted([a]), ts, other, 3)
    second = run_episode(_Scripted([a]), ts, other, 3)
    assert first["success"] and second["success"], "every episode starts from a fresh world: a write made by one episode must not leak into the next"
    ep = run_episode(_Scripted([("get_ticket", {"ticket_id": "T999"})]), ts, task, 3)
    assert ep["errors"] == 1 and not ep["success"], "an error from the toolset is recorded, not raised"
    print("exercise 3 looks right")


if __name__ == "__main__":
    _try("exercise 3", _check_episode)

### A call, an error, a second call

A transcript makes the mechanism concrete. The next cell runs the policy by hand on one task against the all-bad
and the all-good toolsets, using a small copy of the call loop you just wrote. Look at what each error tells
the policy, and what the policy does with it.

In [ ]:
def transcript(levers: tuple, task: dict) -> None:
    ts, world = Toolset(*levers), fresh_world()
    policy, calls = make_policy(ts, task), []
    print(f"-- {levers}  {task['id']} {task['op']}, the user says {task['wording']}")
    for _ in range(MAX_STEPS):
        step = policy.propose(calls)
        if step is None:
            break
        ok, text = ts.call(world, step[0], step[1])
        calls.append({"tool": step[0], "args": step[1], "ok": ok, "text": text})
        shown = str(text) if len(str(text)) < 90 else str(text)[:87] + "..."
        print(f"   {step[0]}({json.dumps(step[1])}) -> {'ok' if ok else 'ERROR'}: {shown}")
    print(f"   judged correct: {judge(world, task)}")


def show_transcripts() -> None:
    task = next(t for t in TASKS if t["op"] == "set_status")
    transcript(("bad", "bad", "bad"), task)
    transcript(("good", "good", "good"), task)


if __name__ == "__main__":
    _try("transcripts", show_transcripts, needs=("exercise 1", "exercise 3"))

In [ ]:
def play_all(levers: tuple) -> list:
    """Every task against one toolset, each episode with its own policy and its own world."""
    ts = Toolset(*levers)
    return [run_episode(make_policy(ts, t), ts, t, MAX_STEPS) for t in TASKS]


def show_one_episode() -> None:
    eps = play_all(("bad", "bad", "good"))
    ep = next(e for e, t in zip(eps, TASKS) if t["op"] == "reassign" and e["errors"])
    print("a recovered or failed episode, from the toolset whose errors are good and whose names and parameters are bad:")
    for c in ep["calls"]:
        print(f"   {c['tool']}({json.dumps(c['args'])}) -> {'ok' if c['ok'] else str(c['text'])[:100]}")
    print("   success:", ep["success"])


if __name__ == "__main__":
    _try("one episode", show_one_episode, needs=("exercise 1", "exercise 3"))

## 6. Exercise 4: four numbers that say what an episode cost

A success rate alone hides how a toolset fails. Two toolsets with the same rate can differ in how many calls
they waste, and in whether their failures announce themselves. **`summarise(episodes)`** takes the dicts that
`run_episode` returns and gives:

* `n`: the number of episodes, and `success`: the fraction that succeeded, as a `Fraction`;
* `wasted`: errors per episode, a `Fraction` (total `errors` over `n`);
* `silent`: the **count** of episodes that did not succeed although they stopped on a call without an error
  (the tool accepted something wrong);
* `gave_up`: the count of episodes that did not stop on a call without an error;
* `recovery`: among episodes with at least one error, the fraction that still succeeded, as a `Fraction`
  (`Fraction(0)` when no episode had an error).

An empty list raises `ValueError`. All fractions are exact: `Fraction(a, b)`, never a float.

<details><summary>💡 Hint 1 — what to think about</summary>

`silent` and `gave_up` are two ways not to succeed, and both are counts. Which two fields of each episode decide
which one it is? What denominators do `success`, `wasted` and `recovery` use, and are they the same?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Start by raising on an empty list. Count with generator expressions over the episodes, and take the recovery
denominator from the episodes with `errors > 0` only, guarding the zero case.

</details>

In [ ]:
def summarise(episodes: list) -> dict:
    """Exact summary of a list of episode dicts (see the fields above).

    Example:
        >>> summarise([{"errors": 0, "success": True, "stopped_ok": True}])["success"]
        Fraction(1, 1)

    Returns:
        A dict with keys n, success, wasted, silent, gave_up, recovery.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_summary() -> None:
    eps = [{"errors": 0, "success": True, "stopped_ok": True}, {"errors": 2, "success": True, "stopped_ok": True},
           {"errors": 1, "success": False, "stopped_ok": True}, {"errors": 3, "success": False, "stopped_ok": False}]
    got = summarise(eps)
    assert got["n"] == 4 and got["success"] == Fraction(1, 2) and isinstance(got["success"], Fraction), f"got success {got['success']!r}: successes over episodes, exact"
    assert got["wasted"] == Fraction(3, 2), f"got {got['wasted']!r}: total errors over ALL episodes, including those with none"
    assert got["silent"] == 1 and got["gave_up"] == 1, f"got silent {got['silent']}, gave_up {got['gave_up']}: silent is failure after a clean-looking stop; gave_up is no clean stop"
    assert got["recovery"] == Fraction(1, 3), f"got {got['recovery']!r}: only episodes that met an error count, and they are three"
    assert summarise([{"errors": 0, "success": True, "stopped_ok": True}])["recovery"] == Fraction(0), "no errors anywhere: recovery is Fraction(0), not an error"
    try:
        summarise([])
    except ValueError:
        pass
    else:
        raise AssertionError("an empty list must raise ValueError")
    print("exercise 4 looks right")


if __name__ == "__main__":
    _try("exercise 4", _check_summary)

In [ ]:
def _pct(x: Fraction) -> str:
    return f"≈{100 * x.numerator / x.denominator:.1f}%"


def show_two_toolsets() -> None:
    rows = {lv: summarise(play_all((lv, lv, lv))) for lv in ("bad", "good")}
    print(f"{'toolset':<28}{'success':>9}{'wasted/task':>13}{'silent':>8}{'gave up':>9}{'recovery':>10}")
    for lv, s in rows.items():
        print(f"{'all ' + lv:<28}{_pct(s['success']):>9}{'≈%.2f' % float(s['wasted']):>13}{s['silent']:>8}{s['gave_up']:>9}{_pct(s['recovery']):>10}")
    n = rows["bad"]["n"]
    print(f"\n(each row is {n} episodes; silent and gave up are counts of those episodes)")
    print("these are results for the scripted policy and its stated tendencies; they are not a rate for any real model")


if __name__ == "__main__":
    _try("two toolsets", show_two_toolsets, needs=("exercise 1", "exercise 3", "exercise 4"))

## 7. Exercise 5: which lever pays most?

The two rows above change several things at once. To say which change did the work, run every combination of
the levers and read the table. (The errors lever bundles validation with wording, as section 1 says, so its effect
is the effect of both together.) A table keyed by `(names, params, errors)` with `0` for bad and `1` for good maps
each combination to its number of successes out of the tasks.

**`main_effects(table)`** returns, for each lever in the order (names, params, errors), a `Fraction`: the average
successes of the four combinations where that lever is good, minus the average of the four where it is bad.
It raises `ValueError` unless the table's keys are exactly the eight combinations: a missing key, and an extra key
as well, are both errors.

**`best_single_fix(table)`** starts from the all-bad corner `(0, 0, 0)` and asks which **one** lever, switched to good
on its own, adds the most successes. It returns `(index, gain)` with the lowest index winning a tie, and `gain`
an integer (it can be zero or negative). When levers interact, the two answers can differ: an average over the
combinations is not what you gain by fixing one thing in a bad setup.

<details><summary>💡 Hint 1 — what to think about</summary>

For the average over the other levers' settings, which keys do you select, and what is the exact way to divide? For the single fix, which key
differs from `(0, 0, 0)` in exactly one place?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Loop over the lever index and over the keys, summing by the value of that position; divide the sums by four with
`Fraction`. For the best fix, build the three neighbouring keys and take the first maximal gain.

</details>

In [ ]:
def main_effects(table: dict) -> tuple:
    """Average effect of switching each lever to good, over the other levers' four settings.

    Example:
        >>> t = {k: 10 * k[0] for k in [(a, b, c) for a in (0, 1) for b in (0, 1) for c in (0, 1)]}
        >>> main_effects(t)
        (Fraction(10, 1), Fraction(0, 1), Fraction(0, 1))

    Returns:
        A tuple of three Fractions.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def best_single_fix(table: dict) -> tuple:
    """(lever index, gain in successes) of the best single change away from the all-bad toolset; lowest index wins ties.

    Example:
        >>> t = {k: 5 * k[1] for k in [(a, b, c) for a in (0, 1) for b in (0, 1) for c in (0, 1)]}
        >>> best_single_fix(t)
        (1, 5)

    Returns:
        (index, gain) with index in 0..2.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_effects() -> None:
    keys = [(a, b, c) for a in (0, 1) for b in (0, 1) for c in (0, 1)]
    flat = {k: 3 * k[0] + 7 * k[2] for k in keys}
    assert main_effects(flat) == (Fraction(3), Fraction(0), Fraction(7)), f"got {main_effects(flat)}: an additive table gives each lever's own step"
    synergy = {k: 20 if k == (1, 1, 1) else 0 for k in keys}
    got = main_effects(synergy)
    assert got == (Fraction(5), Fraction(5), Fraction(5)) and all(isinstance(g, Fraction) for g in got), (
        f"got {got}: divide by four exactly, with Fraction, not a float or integer division")
    assert best_single_fix(synergy) == (0, 0), "when only the full set of fixes helps, no single fix gains anything; the lowest index wins the tie"
    skew = {k: 10 * k[1] - 4 * k[0] for k in keys}
    assert best_single_fix(skew) == (1, 10) and main_effects(skew)[0] == Fraction(-4), "a lever can hurt: a negative effect stays negative"
    try:
        main_effects({k: 1 for k in keys[:7]})
    except ValueError:
        pass
    else:
        raise AssertionError("a table with a missing combination must raise ValueError")
    print("exercise 5 looks right")


if __name__ == "__main__":
    _try("exercise 5", _check_effects)

### The measurement

Now run all the combinations. Read the table by row, then the two summaries underneath. Where `main_effects`
and `best_single_fix` disagree about the lever, the levers interact, and the table says how.

In [ ]:
def _measure() -> None:
    table, summaries = {}, {}
    for a in (0, 1):
        for b in (0, 1):
            for c in (0, 1):
                s = summarise(play_all(tuple("good" if x else "bad" for x in (a, b, c))))
                summaries[(a, b, c)] = s
                table[(a, b, c)] = int(s["success"] * s["n"])
    n = summaries[(0, 0, 0)]["n"]
    print(f"successes out of {n} episodes (1 = the good version of the lever)")
    print(f"{'names':>6}{'params':>8}{'errors':>8}{'success':>9}{'wasted/task':>13}{'silent':>8}{'gave up':>9}{'recovery':>10}")
    for k, s in summaries.items():
        print(f"{k[0]:>6}{k[1]:>8}{k[2]:>8}{table[k]:>9}{'≈%.2f' % float(s['wasted']):>13}{s['silent']:>8}{s['gave_up']:>9}{_pct(s['recovery']):>10}")
    effects = main_effects(table)
    index, gain = best_single_fix(table)
    lever = ("names", "params", "errors")
    print("\naverage effect of making each lever good (successes out of", n, "):")
    for name, e in zip(lever, effects):
        print(f"  {name:<8}{'+' if e >= 0 else '-'}{abs(float(e)):.2f}  (exact {e})")
    print(f"\nthe best single fix from the all-bad toolset is {lever[index]}: {'+' if gain >= 0 else ''}{gain} successes")
    top = max(range(3), key=lambda i: effects[i])
    print(f"the largest average effect belongs to {lever[top]}; "
          + ("the same lever wins both ways." if top == index else "it is not the best single fix, so the levers interact."))
    top_score = max(table.values())
    print(f"the best score, {top_score} of {n}, is reached by {sorted(k for k in table if table[k] == top_score)}")
    good_wasted = float(summaries[(1, 1, 1)]["wasted"])
    if table[(1, 1, 1)] == n:
        print(f"the all-good toolset succeeds on every task, with ≈{good_wasted:.2f} wasted calls per task: the policy's slips still\n"
              "reach it, and its corrective errors repair them. That is the script's ceiling and not a prediction; a real model has\n"
              "failure modes this script lacks")
    print("\nthe errors lever switches validation on as well as improving the wording, so its effect is both together. The\n"
          "policy's tendencies are assumptions, so read these as which parts of a specification the stated rules punish.\n"
          "Testing a real model would need a policy of your own whose propose(calls) asks the model and returns (tool, args),\n"
          "run through the loop of exercise 3 on the same toolsets and tasks.")


if __name__ == "__main__":
    _try("measurement", _measure, needs=("exercise 1", "exercise 3", "exercise 4", "exercise 5"))

## 8. What this measurement tells you, and what it does not

* **It tells you about mechanics.** Which parts of a specification a reader with stated habits trips on, and how
  much an error message that names the fix is worth to a reader that acts on such messages. The policy acts on
  a message only because its script was written to; a real model may read more or less from the same text.
* **It does not give a rate.** The tendencies and their rates, the wordings users use, and the way the bad
  specifications were written are the author's choices. A different author would get different numbers. The
  toolsets and tasks were fixed before any run and no seed was searched, but that makes the numbers repeatable,
  not representative.
* **The vendor guidance is advice, not a result of this lesson.** The sources section quotes what Anthropic's
  documents say about tool descriptions, parameter names, consolidation and error responses, each read on its access
  date. Nothing printed above confirms or refutes it; to test it on a model you would write a policy whose
  `propose(calls)` asks the model and returns `(tool, args)`, and run it through the loop of exercise 3, as this
  lesson runs the script. T07-L01 and T07-L03 build the loop and the metrics, not a path that calls a model.
* **The errors lever is two levers.** Good errors validate as well as explain; bad errors let a wrong value through,
  and the silent column counts those. The table cannot separate validation from wording, and the policy reads only
  the author's message shape.
* **The extremes are built in.** In the bad toolsets some operations cannot be finished by any reader, because a
  value they need (`assignee` for the update tool's `field`) appears nowhere in the specification. In the all-good
  toolset the corrective errors repair the policy's slips. The floor and the ceiling show information absent or
  present, not ambiguity.
* **Consolidation was held fixed.** Both specifications have the same tools doing the same pairs of jobs. Whether
  many narrow tools beat a few wide ones is a question this lesson does not ask.

## 9. Common mistakes

* **Reading a rate off a scripted policy.** The table is a property of the script and the specifications together.
* **Counting only errors.** A tool that accepts a wrong value shows no error at all; the silent column is where it
  appears. A success rate and a wasted-call count together can still look fine while it hides there.
* **Making an error say "invalid".** An error that does not name the parameter, the value and what is accepted gives
  a reader nothing to repair, and the recovery column shows the cost.
* **Mixing a pattern into the description only.** The id pattern in this lesson lives in the specification as data
  as well as in prose, so a validator can check it.
* **Taking an average effect for a recipe.** `main_effects` averages over settings you may not be in.

## 10. What this lesson does not cover

* **Real models.** No language model is called; the policy is a script.
* **Response design.** What a tool returns (concise or detailed output, pagination, truncation) is not varied.
* **Selection among many tools.** Each toolset is small; the cost of choosing among dozens, and of namespacing, is not measured.
* **Security.** A tool description is also an attack surface; T07-L08 covers vetting.

In [ ]:
# Facts about the toolsets you just built, printed by the code that runs here, for the self-check below.
if __name__ == "__main__":
    print(f"{len(KINDS)} tools per toolset, {len(OPS)} operations, {len(TASKS)} tasks, at most {MAX_STEPS} calls per episode")
    _levers = len(("names", "params", "errors"))
    print("levers crossed:", _levers, "-> toolsets:", 2 ** _levers)
    print("each episode's policy is seeded from the toolset's levers and the task id, so one toolset's results never depend on another's")

## 11. Self-check

1. Suppose a table like this one showed the toolset with corrective error text recovering far more often than the one with vague error text. What could you conclude?
   - (a) A real model would recover from corrective errors at the same rate, because the table was computed exactly and its seeds are fixed.
   - (b) For a reader that acts on such messages, as the script does, naming the fix is worth a gain; whether a model acts on them needs its own run.
   - (c) Corrective error text matters only for tool names, because the script never reads a message when a parameter is wrong.
   - (d) Vague errors would be better, because they push the agent to try other tools, and trying another tool is how a stuck agent recovers.

2. A tool answers an unknown person name with an empty list and no error. Why is that worse than an error?
   - (a) The agent retries an empty list until the step cap, so the episode always ends in giving up.
   - (b) Empty lists use more tokens than error strings, so the episode costs more.
   - (c) The agent takes the empty list for a finished job and stops; the failure shows in no error count.
   - (d) Empty lists take longer to produce than error strings, so the wasted-call count rises.

3. Why is each episode's random generator seeded from the toolset's levers and the task id, instead of using one generator for the whole run?
   - (a) So that run order cannot change what an episode does, and any one episode can be replayed alone.
   - (b) Because a single generator would give every episode the same sequence of coin flips, so every tie would break the same way.
   - (c) Because Python's random module cannot be seeded more than once in a process, so each episode needs its own object.
   - (d) Because the policy needs the task id to know which tool is the correct one for the job.

4. Suppose `best_single_fix` and `main_effects` name different levers. The most likely reason is:
   - (a) One of the two functions has a bug, since both measure the same thing, and the lesson's table cannot tell you which.
   - (b) The table has too few episodes for averages to mean anything, so the larger of the two is noise.
   - (c) Averages over several settings are always larger than a single change, so they pick the lever with more settings.
   - (d) The levers interact: an average over all eight settings is not the gain from changing one lever in the all-bad toolset.

Answers, with reasoning, come with this lesson's worked solution when you enrol on Synapsa.

## 12. Sources, with the date each was read
Every dated statement above is a `claims.yaml` entry with its source, a verbatim quote and the day it was read
(19 and 20 October 2026, dated per entry in `claims.yaml`). Vendor guidance changes often: read the pages again before you rely on them.
* Anthropic, "Building effective agents" (19 December 2024): the agent-computer interface (ACI), crafted through tool documentation and testing.
* Anthropic, "Writing effective tools for agents" (11 September 2025): unambiguous parameter names, actionable error responses, consolidated operations.
* Anthropic, Claude API documentation, "Define tools": detailed descriptions, fewer and more capable tools.

In [ ]:
# Your progress board: one line per exercise, from the checks you have run, then the count.
_MARKS = {"passed": "✅ passed     ", "failed": "❌ failed     ", "not started": "⏳ not started"}


def _progress_board() -> None:
    """One line per exercise, from what the checks recorded, then how many are complete."""
    print("\nyour progress")
    done, next_up = 0, None
    for label, shown, stubs, where in _EXERCISES:
        status = _STATUS.get(label, "not started")
        done += status == "passed"
        waiting = _WAITING_ON.get(label)
        note = (f"   (waiting on `{waiting}`)" if status == "not started" and waiting not in (None, "") and waiting not in stubs
                else "")
        print(f"  {_MARKS[status]}  {label} · {shown}{note}")
        if status != "passed" and next_up is None:
            next_up = (label, shown, where)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    if next_up:
        print(f"next: {next_up[0]}, {next_up[1]}, in {next_up[2]}. Its cell has hints if you are stuck.")


def _finish() -> None:
    """The board, then the verdict. A stub you have not reached yet is not a failure; a check that ran and
    came back wrong is. In a script or under CI that ends the run non-zero; in a notebook kernel it is a printed line."""
    _progress_board()
    failed = [label for label, status in _STATUS.items() if status == "failed"]
    if failed and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(failed))
    if failed:
        print("checks failed: " + ", ".join(failed) + " — each one printed its likely mistake where it ran.")


if __name__ == "__main__":
    _finish()

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.